# MedCLIP Evaluation
**Datasets required (add in Kaggle sidebar):**
- `maxzhang646/medclip-checkpoint` — our trained checkpoint
- `nih-chest-xrays/data` — NIH ChestX-ray14 for zero-shot eval
- `raddar/chest-xrays-indiana-university` — OpenI for retrieval eval

In [ ]:
# Install dependencies
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q transformers scikit-learn

In [ ]:
# Clone repo to get src/
!git clone -q https://github.com/maxzhang646/medical-clip.git
import sys
sys.path.insert(0, 'medical-clip/src')

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.metrics import roc_auc_score

from model import MedCLIP
from dataset import OpenIDataset, NIHDataset, NIH_CLASSES
from prompts import build_prompts
from loss import infonce_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Paths
CKPT_PATH   = '/kaggle/input/medclip-checkpoint/best.pt'
NIH_DIR     = '/kaggle/input/data'
INDIANA_DIR = '/kaggle/input/chest-xrays-indiana-university'

# Verify paths
for p in [CKPT_PATH, NIH_DIR, INDIANA_DIR]:
    print(f'{p}: {"OK" if os.path.exists(p) else "MISSING"}')

In [ ]:
# Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained('medicalai/ClinicalBERT')
model = MedCLIP().to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()
print('Model loaded.')

## 1. Retrieval Evaluation (OpenI test split)

In [ ]:
def recall_at_k(sim_matrix, k_list):
    n = sim_matrix.shape[0]
    ranks = np.argsort(-sim_matrix, axis=1)
    gt = np.arange(n)
    results = {}
    for k in k_list:
        hits = np.any(ranks[:, :k] == gt[:, None], axis=1)
        results[f'R@{k}'] = hits.mean() * 100
    results['MedR'] = float(np.median(
        [np.where(ranks[i] == i)[0][0] + 1 for i in range(n)]
    ))
    return results

@torch.no_grad()
def run_retrieval(model, loader, device):
    img_embs, txt_embs = [], []
    for batch in loader:
        img_embs.append(model.encode_image(batch['image'].to(device)).cpu())
        txt_embs.append(model.encode_text(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device)
        ).cpu())
    I = torch.cat(img_embs).numpy()
    T = torch.cat(txt_embs).numpy()
    sim = I @ T.T
    return {'I→T': recall_at_k(sim, [1, 5, 10]),
            'T→I': recall_at_k(sim.T, [1, 5, 10])}

test_ds = OpenIDataset(INDIANA_DIR, tokenizer, split='test')
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)
print(f'Test samples: {len(test_ds)}')

results = run_retrieval(model, test_loader, device)
print('\n=== Retrieval Results (MedCLIP) ===')
for direction, metrics in results.items():
    print(f'\n{direction}')
    for k, v in metrics.items():
        print(f'  {k}: {v:.2f}')

In [ ]:
# Baseline: vanilla CLIP (no fine-tuning)
import clip
from PIL import Image
from torchvision import transforms

clip_model, clip_preprocess = clip.load('ViT-B/32', device=device)
clip_model.eval()

@torch.no_grad()
def run_retrieval_vanilla_clip(test_ds, device):
    img_embs, txt_embs = [], []
    for s in test_ds.samples:
        img = clip_preprocess(Image.open(s['image_path']).convert('RGB')).unsqueeze(0).to(device)
        img_embs.append(clip_model.encode_image(img).float().cpu())
        txt = clip.tokenize([s['caption'][:77]], truncate=True).to(device)
        txt_embs.append(clip_model.encode_text(txt).float().cpu())
    I = torch.nn.functional.normalize(torch.cat(img_embs), dim=-1).numpy()
    T = torch.nn.functional.normalize(torch.cat(txt_embs), dim=-1).numpy()
    sim = I @ T.T
    return {'I→T': recall_at_k(sim, [1, 5, 10]),
            'T→I': recall_at_k(sim.T, [1, 5, 10])}

baseline_results = run_retrieval_vanilla_clip(test_ds, device)
print('\n=== Retrieval Results (Vanilla CLIP baseline) ===')
for direction, metrics in baseline_results.items():
    print(f'\n{direction}')
    for k, v in metrics.items():
        print(f'  {k}: {v:.2f}')

## 2. Zero-shot Classification (NIH ChestX-ray14)

In [ ]:
DISEASES = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion', 'Infiltration', 'Pneumonia', 'Pneumothorax'
]

@torch.no_grad()
def encode_prompts(model, tokenizer, diseases, prompt_key, device):
    embs = []
    for d in diseases:
        prompts = build_prompts(d)[prompt_key]
        enc = tokenizer(prompts, padding=True, truncation=True,
                        max_length=128, return_tensors='pt').to(device)
        e = model.encode_text(enc['input_ids'], enc['attention_mask'])
        embs.append(e.mean(dim=0))
    return torch.stack(embs)

@torch.no_grad()
def run_zeroshot(model, tokenizer, loader, diseases, prompt_key, device):
    text_embs = encode_prompts(model, tokenizer, diseases, prompt_key, device)
    all_logits, all_labels = [], []
    for batch in loader:
        img_emb = model.encode_image(batch['image'].to(device))
        logits  = img_emb @ text_embs.T
        all_logits.append(logits.cpu().numpy())
        all_labels.append(batch['labels'].numpy())
    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    aucs = {}
    for i, cls in enumerate(diseases):
        if labels[:, i].sum() > 0:
            aucs[cls] = roc_auc_score(labels[:, i], logits[:, i])
    aucs['Macro AUC'] = np.mean(list(aucs.values()))
    return aucs

nih_ds = NIHDataset(NIH_DIR, classes=DISEASES)
nih_loader = DataLoader(nih_ds, batch_size=256, shuffle=False, num_workers=4)
print(f'NIH samples: {len(nih_ds)}')

In [ ]:
# Run all prompt variants
PROMPT_KEYS = ['simple', 'findings', 'clinical', 'patient', 'radiologist', 'ensemble']
all_results = {}

for pk in PROMPT_KEYS:
    aucs = run_zeroshot(model, tokenizer, nih_loader, DISEASES, pk, device)
    all_results[pk] = aucs
    print(f'[{pk}] Macro AUC: {aucs["Macro AUC"]:.4f}')

In [ ]:
# Full results table
df = pd.DataFrame(all_results).T
print('\n=== Zero-shot AUC by prompt ===')
print(df.round(4).to_string())

## 3. Visualization

In [ ]:
# Prompt ablation bar chart
macro_aucs = {k: all_results[k]['Macro AUC'] for k in PROMPT_KEYS}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(macro_aucs.keys(), macro_aucs.values(), color='steelblue', width=0.6)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
ax.set_ylim(0.5, 0.8)
ax.set_ylabel('Macro AUC')
ax.set_title('Zero-shot AUC by Prompt Template (NIH 8 classes)')
plt.tight_layout()
plt.savefig('prompt_ablation.png', dpi=150)
plt.show()

In [ ]:
# Retrieval comparison table
print('=== Retrieval Comparison ===')
rows = []
for direction in ['I→T', 'T→I']:
    for metric in ['R@1', 'R@5', 'R@10', 'MedR']:
        rows.append({
            'Direction': direction,
            'Metric': metric,
            'MedCLIP': results[direction][metric],
            'Vanilla CLIP': baseline_results[direction][metric]
        })
df_ret = pd.DataFrame(rows)
print(df_ret.to_string(index=False))